# 13주차 — 실전 — 코호트 추출과 Table 1
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 연구 질문
**"2020년 1월 1일 이후 처음으로 뇌경색(I63)을 진단받은 환자"** 코호트를 만들고 Table 1까지 완성한다.

## 1단계 — 후보군: I63 이력자 전체와 첫 진단일

In [ ]:
# 첫 진단일 (washout 적용 전 — 10명)
q("""
SELECT v.patient_id, MIN(v.visit_date) AS first_dx_date
FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'I63%'
GROUP BY v.patient_id ORDER BY first_dx_date
""")

## 2단계 — washout: '첫' 진단이 2020년 이후인 신규 환자만

In [ ]:
# index date 확정 (8명)
q("""
SELECT v.patient_id, MIN(v.visit_date) AS index_date
FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'I63%'
GROUP BY v.patient_id
HAVING MIN(v.visit_date) >= '2020-01-01'
ORDER BY index_date
""")

## 3단계 — WITH(CTE)로 특성 붙이기

In [ ]:
# 코호트 + 성별 + index 시점 나이
q("""
WITH stroke_cohort AS (
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
  WHERE d.kcd_code LIKE 'I63%'
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01')
SELECT c.patient_id, c.index_date, p.sex,
       CAST(substr(c.index_date,1,4) AS INT)
         - CAST(substr(p.birth_date,1,4) AS INT) AS age_at_index
FROM stroke_cohort c JOIN patient p ON p.patient_id = c.patient_id
""")

## 4단계 — 동반질환 플래그 (EXISTS)

In [ ]:
# 고혈압(I10) 동반 여부
q("""
WITH stroke_cohort AS (
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
  WHERE d.kcd_code LIKE 'I63%'
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01')
SELECT c.patient_id, c.index_date,
  CASE WHEN EXISTS (
    SELECT 1 FROM visit v2 JOIN diagnosis d2 ON d2.visit_id = v2.visit_id
    WHERE v2.patient_id = c.patient_id AND d2.kcd_code LIKE 'I10%')
  THEN 1 ELSE 0 END AS htn_comorbid
FROM stroke_cohort c
""")

## 5단계 — Table 1

In [ ]:
# 성별·평균나이 요약
q("""
WITH stroke_cohort AS (
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON d.visit_id = v.visit_id
  WHERE d.kcd_code LIKE 'I63%'
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01')
SELECT p.sex AS 성별, COUNT(*) AS 명수,
  ROUND(AVG(CAST(substr(c.index_date,1,4) AS INT)
    - CAST(substr(p.birth_date,1,4) AS INT)), 1) AS 평균나이
FROM stroke_cohort c JOIN patient p ON p.patient_id = c.patient_id
GROUP BY p.sex
""")

## 프로젝트 스타터 — 이 뼈대를 복사해 조별 주제로 바꾸세요

In [ ]:
# [프로젝트 템플릿]
COHORT_SQL = """
WITH my_cohort AS (
  -- ① 포함 기준: 코드/기간을 조별 주제로 교체
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON d.visit_id = v.visit_id
  WHERE d.kcd_code LIKE '__%'          -- ← 여기
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01'  -- ← washout 기준
)
SELECT c.*, p.sex FROM my_cohort c
JOIN patient p ON p.patient_id = c.patient_id
"""
# q(COHORT_SQL)  # 주석을 풀어 실행

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**프로젝트 체크리스트**
```sql
-- ① 포함 기준(코드·기간)과 제외 기준을 정의서에 먼저 글로 쓴다
-- ② 단계마다 대상자 수를 주석으로 남긴다 (attrition)
-- ③ Table 1 변수는 성별·연령 + 조별 선택 2개 이상
```